# Yellow Taxi Trip Data Analysis

**NYC Yellow Taxi — January 2025**

A complete Python data-analysis workflow covering inspection, cleaning, feature engineering, statistical analysis, visualization, route analysis, business insights, limitations, and conclusion.


# **DATASET DESCRIPTION**

NYC Yellow Taxi Trip Data for January 2025, stored in Parquet format. It contains trip-level information such as pickup/drop-off time, passenger count, trip distance, locations, vendors, rate codes, and payment types.

# **OBJECTIVE AND ANALYSIS**

To perform Exploratory Data Analysis (EDA) using Python to clean the data, identify patterns, analyze trip characteristics, create meaningful visualizations, and generate evidence-based operational insights.


The analysis will cover data quality, trip duration and distance, time-based demand, passenger patterns, pickup/drop-off locations, routes, payment types, vendors, rate codes, and correlations, followed by key findings and limitations.

**IMPORTING LIBRARIES**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

**LOADING THE DATASET**

In [ ]:
df=pd.read_parquet('/content/yellow_tripdata_2025-01.parquet')
print('='*60)
print('DATASET LOADED SUCCESSFULLY')
print('='*60)
print(f'Shape of the dataset:              {df.shape}')
print(f'No of rows present in dataset      {df.shape[0]}')
print(f'No of columns present in dataset   {df.shape[1]}')

**DATA** **INSPECTION**

In [ ]:
print('='*60)
print('FIRST 10 ROWS OF DATASET')
print('='*60)
df.head(10)

In [ ]:
print('='*60)
print('LAST 10 ROWS OF DATASET')
print('='*60)
df.tail(10)

In [ ]:
print('='*60)
print('COLUMNS OF DATASET')
print('='*60)
index=1
for i in df.columns:
  print(index,'.',i)
  index += 1

In [ ]:
print('='*60)
print('DATASET INFORMATION')
print('='*60)
df.info()

In [ ]:
print('='*60)
print('DATATYPES PRESENT IN DATASET')
print('='*60)
df.dtypes

In [ ]:
print('='*60)
print('STATISTICAL INFORMATION OF THE DATASET')
print('='*60)
df.describe()

In [ ]:
print('='*60)
print('STATISTICAL INFORMATION OF OBJECTS')
print('='*60)
df.describe(include=object)

**HANDLING MISSING VALUES**

In [ ]:
print('='*60)
print('MISSING VALUE ANALYSIS')
print('='*60)
missing_values=df.isnull().sum()
missing_percentage=missing_values/len(df)*100
missing_df=pd.DataFrame({'Missing values':missing_values,'Missing perentage':missing_percentage})
missing_df

print(f'Missing values present in different columns:\n{missing_values}  ')
print()
print(f'Missing values percentage:\n{missing_percentage}')

In [ ]:
print('='*60)
print('DATA CLEANING')
print('='*60)
print('\n1. DATASET SHAPE')
print('Rows:',df.shape[0])
print('Columns:',df.shape[1])

print('\n2. MISSING VALUES')
missing=df.isnull().sum()
missing_pct=(missing/len(df))*100
missing_report=pd.DataFrame({'Missing Values':missing,'Missing %':missing_pct})
display(missing_report[missing_report['Missing Values']>0])

print('\n3. DUPLICATE ROWS')
print('Duplicates:',df.duplicated().sum())

print('\n4. DATA TYPES')
display(df.dtypes)

print('\n5. UNIQUE VALUES')
display(df.nunique())

print('\n6. LOCATION ID RANGE')
print('PULocationID:',df['PULocationID'].min(),'-',df['PULocationID'].max())
print('DOLocationID:',df['DOLocationID'].min(),'-',df['DOLocationID'].max())

print('\n7. CATEGORICAL VALUES')
print('VendorID:',sorted(df['VendorID'].dropna().unique()))
print('RatecodeID:',sorted(df['RatecodeID'].dropna().unique()))
print('store_and_fwd_flag:',df['store_and_fwd_flag'].dropna().unique())
print('payment_type:',sorted(df['payment_type'].dropna().unique()))

print('\n8. SUSPICIOUS RECORDS')
print('Zero/negative distance:',(df['trip_distance']<=0).sum())
print('Zero/negative passenger count:',(df['passenger_count']<=0).sum())
print('Drop-off before pickup:',(df['tpep_dropoff_datetime']<df['tpep_pickup_datetime']).sum())
print('Zero duration:',(df['tpep_dropoff_datetime']==df['tpep_pickup_datetime']).sum())

duration=((df['tpep_dropoff_datetime']-df['tpep_pickup_datetime']).dt.total_seconds()/60)
print('Duration > 24 hours:',(duration>1440).sum())

print('\n9. EXTREME VALUES')
print('Maximum trip distance:',df['trip_distance'].max())
print('Maximum trip duration:',duration.max())

print('\n10. PASSENGER COUNT DISTRIBUTION')
display(df['passenger_count'].value_counts(dropna=False).sort_index())

print('\n11. RATECODEID DISTRIBUTION')
display(df['RatecodeID'].value_counts(dropna=False).sort_index())

print('\n12. PAYMENT TYPE DISTRIBUTION')
display(df['payment_type'].value_counts(dropna=False).sort_index())

In [ ]:
clean_df=df.copy()
clean_df=clean_df.drop_duplicates()
clean_df=clean_df[clean_df['tpep_dropoff_datetime']>=clean_df['tpep_pickup_datetime']]
clean_df=clean_df[clean_df['trip_distance']>0]
clean_df=clean_df[clean_df['passenger_count']>0]
clean_df['trip_duration_minutes']=(
    clean_df['tpep_dropoff_datetime']-
    clean_df['tpep_pickup_datetime']).dt.total_seconds()/60
clean_df=clean_df[(clean_df['trip_duration_minutes']>0)&(clean_df['trip_duration_minutes']<=1440)]
print('='*70)
print('DATA CLEANING COMPLETED')
print('='*70)
print('Original rows:',len(df))
print('Cleaned rows:',len(clean_df))
print('Rows removed:',len(df)-len(clean_df))
print('Original columns:',df.shape[1])
print('Cleaned columns:',clean_df.shape[1])
print('Remaining missing values:',clean_df.isnull().sum().sum())
print('Remaining duplicates:',clean_df.duplicated().sum())

**DUPLICATE VALUE ANALYSIS**

In [ ]:
print('='*60)
print('DUPLICATE VALUE ANALYSIS')
print('='*60)
duplicate_values=df.duplicated().sum()
duplicate_values_percentage=(duplicate_values/len(df))*100
print(f'Number of duplicate rows: {duplicate_values}')
print(f'Percentage of duplicate rows: {duplicate_values_percentage:.2f}%')


# **FEATURE ENGINEERING**

**FEATURE ENGINEERING FOR TRIP DURATION MINUTES**

In [ ]:
print('='*60)
print('FEATURE ENGINEERING FOR TRIP DURATION MINUTES')
print('='*60)
def categories(duration):
  if duration<5:
    return 'Very Short'
  elif duration<15:
    return 'Short'
  elif duration<30:
    return 'Medium'
  elif duration<60:
    return 'Long'
  else:
    return 'Very Long'
clean_df['trip_duration_category']=clean_df['trip_duration_minutes'].apply(categories)
duration_count=clean_df['trip_duration_category'].value_counts()
print('The trip duration distribution among the trips are:')
category=['Very Short','Short','Medium','Long','Very Long']
for i in category:
  print(f'{i} : {duration_count[i]}')


print('='*60)
print('FEATURE ENGINEERING FOR PICKUP DATE')
print('='*60)
clean_df['pickup_date']=clean_df['tpep_pickup_datetime'].dt.date
print(clean_df['pickup_date'].head())




print('='*60)
print('FEATURE ENGINEERING FOR PICKUP HOUR')
print('='*60)
clean_df['pickup_hour']=clean_df['tpep_pickup_datetime'].dt.hour
print(clean_df['pickup_hour'].head())


print('='*60)
print('FEATURE ENGINEERING FOR DAY OF THE WEEK')
print('='*60)
clean_df['day_of_week']=clean_df['tpep_pickup_datetime'].dt.day_name()
print(clean_df[['tpep_pickup_datetime','day_of_week']].head())


print('='*60)
print('FEATURE ENGINEERING FOR WEEKEND INDICATOR')
print('='*60)
clean_df['is_weekend']=clean_df['day_of_week'].isin(['Saturday','Sunday'])
print(clean_df[['day_of_week','is_weekend']].head())

print('='*60)
print('FEATURE ENGINEERING ON AVERAGE SPEED')
print('='*60)
clean_df['average_speed_mph']=clean_df['trip_distance']/(clean_df['trip_duration_minutes']/60)
print(clean_df[['trip_distance','trip_duration_minutes','average_speed_mph']].head())

print('='*60)
print('FEATURE ENGINEERING ON DISTANCE CATEGORY')
print('='*60)
def categories(distance):
  if distance<5:
    return 'Short'
  elif distance<10:
    return 'Medium'
  else:
    return 'Long'
clean_df['distance_category']=clean_df['trip_distance'].apply(categories)
distance_count=clean_df['distance_category'].value_counts()
print('The trip distance distribution is:')
category=['Short','Medium','Long']
for i in category:
  print(f'{i} : {distance_count[i]}')

print('='*60)
print('FEATURE ENGINEERING ON DURATION CATEGORY')
print('='*60)
def categories(duration):
  if duration<5:
    return 'Very Short'
  elif duration<15:
    return 'Short'
  elif duration<30:
    return 'Medium'
  else:
    return 'Long'
clean_df['duration_category']=clean_df['trip_duration_minutes'].apply(categories)
duration_count=clean_df['duration_category'].value_counts()
print('The trip duration distribution is:')
category=['Very Short','Short','Medium','Long']
for i in category:
  print(f'{i} : {duration_count[i]}')

print('='*60)
print('FEATURE ENGINEERING ON ROUTE')
print('='*60)
clean_df['route']=clean_df['PULocationID'].astype(str)+' → '+clean_df['DOLocationID'].astype(str)
print(clean_df[['PULocationID','DOLocationID','route']].head())


In [ ]:
features=['trip_duration_minutes','pickup_date','pickup_hour','day_of_week','is_weekend','average_speed_mph','distance_category','duration_category','route']

print(clean_df[features].head())

In [ ]:
print('Final Shape:',clean_df.shape)

In [ ]:
clean_df.to_parquet('yellow_taxi_cleaned_feature_engineered.parquet',index=False)

# **UNIVARIATE ANALYSIS**

**UNIVARIATE ANALYSIS ON TRIP DISTANCE**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON TRIP DISTANCE')
print('='*60)
print('Mean:',clean_df['trip_distance'].mean())
print('Median:',clean_df['trip_distance'].median())
print('Standard deviation:',clean_df['trip_distance'].std())
print('\nPercentiles:')
print(clean_df['trip_distance'].quantile([0.25,0.50,0.75,0.90,0.95,0.99]))
print('\nMinimum:',clean_df['trip_distance'].min())
print('Maximum:',clean_df['trip_distance'].max())

**CHECKING THE OUTLIERS**

In [ ]:
Q1=clean_df['trip_distance'].quantile(0.25)
Q3=clean_df['trip_distance'].quantile(0.75)
IQR=Q3-Q1
lower=Q1-1.5*IQR
upper=Q3+1.5*IQR
outliers=clean_df[(clean_df['trip_distance']<lower)|(clean_df['trip_distance']>upper)]
print('Lower limit:',lower)
print('Upper limit:',upper)
print('Number of outliers:',len(outliers))

**UNIVARIATE ANALYSIS ON TRIP DURATION**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON TRIP DURATION')
print('='*60)
print('Mean:',clean_df['trip_duration_minutes'].mean())
print('Median:',clean_df['trip_duration_minutes'].median())
print('Minimum:',clean_df['trip_duration_minutes'].min())
print('Maximum:',clean_df['trip_duration_minutes'].max())

**CHECKING THE OUTLIERS**

In [ ]:
Q1=clean_df['trip_duration_minutes'].quantile(0.25)
Q3=clean_df['trip_duration_minutes'].quantile(0.75)
IQR=Q3-Q1
lower=Q1-1.5*IQR
upper=Q3+1.5*IQR
outliers=clean_df[(clean_df['trip_duration_minutes']<lower)|(clean_df['trip_duration_minutes']>upper)]
print('\nLower Limit:',lower)
print('Upper Limit:',upper)
print('Number of Outliers:',len(outliers))

**UNIVARIATE ANALYSIS ON PASSENGER COUNT**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON PASSENGER COUNT')
print('='*60)
passenger_count=clean_df['passenger_count'].value_counts().sort_index()
print('Passenger Count Distribution:')
print(passenger_count)
print('\nMost Common Passenger Count:',clean_df['passenger_count'].mode()[0])
print('Missing Passenger Count:',clean_df['passenger_count'].isnull().sum())
plt.figure(figsize=(10,6))
plt.bar(passenger_count.index.astype(str),passenger_count.values,color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Passenger count')
plt.ylabel('Number of trips')
plt.title('Passenger Count Distribution')
plt.show()

**UNIVARIATE ANALYSIS ON PICKUP HOUR**

In [ ]:
print('='*60)
print('Univariate Analysis on Pickup Hour')
print('='*60)
hour_count=clean_df['pickup_hour'].value_counts().sort_index()
print('Trips by Hour:')
print(hour_count)
print('\nHighest Trip Activity:',hour_count.idxmax())
print('Lowest Trip Activity:',hour_count.idxmin())
plt.figure(figsize=(10,6))
plt.bar(hour_count.index,hour_count.values,color='yellow',width=0.7)
plt.xticks(range(24),rotation=45)
plt.xlabel('Pickup Hour')
plt.ylabel('Number of Trips')
plt.title('Trips by Pickup Hour')
plt.show()

**UNIVARIATE ANALYSIS ON DAY OF THE WEEK**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON DAY OF THE WEEK')
print('='*60)
days=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_count=clean_df['day_of_week'].value_counts()
for day in days:
  print(f'{day} : {day_count[day]}')
plt.figure(figsize=(10,6))
plt.bar(days,[day_count[day] for day in days],color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.title('Trips by Day of Week')
plt.show()

**UNIVARIATE ANALYSIS ON PAYMENT TYPE**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON PAYMENT TYPE')
print('='*60)
payment_count=clean_df['payment_type'].value_counts().sort_index()
print('Payment Type Distribution:')
print(payment_count)
plt.figure(figsize=(10,6))
plt.bar(payment_count.index.astype(str),payment_count.values,color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Payment Type')
plt.ylabel('Number of Trips')
plt.title('Payment Type Distribution')
plt.show()

**UNIVARIATE ANALYSIS ON VENDOR**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON VENDOR')
print('='*60)
vendor_count=clean_df['VendorID'].value_counts().sort_index()
print('Trips by Vendor:')
print(vendor_count)
plt.figure(figsize=(10,6))
plt.bar(vendor_count.index.astype(str),vendor_count.values,color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Vendor ID')
plt.ylabel('Number of Trips')
plt.title('Trips by Vendor')
plt.show()

**UNIVARIATE ANALYSIS ON RATE CODE**

In [ ]:
print('='*60)
print('UNIVARIATE ANALYSIS ON RATE CODE')
print('='*60)
rate_count=clean_df['RatecodeID'].value_counts().sort_index()
print('Rate Code Distribution:')
print(rate_count)
plt.figure(figsize=(10,6))
plt.bar(rate_count.index.astype(str),rate_count.values,color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Rate Code')
plt.ylabel('Number of Trips')
plt.title('Rate Code Distribution')
plt.show()

### **TIME BASED ANALYSIS**

In [ ]:
print('='*60)
print('Trips by Hour')
print('='*60)
hour_count=clean_df['pickup_hour'].value_counts().sort_index()
print('Trips by Hour:')
print(hour_count)
print('\nHighest Trip Hour:',hour_count.idxmax())
print('Highest Number of Trips:',hour_count.max())
print('Lowest Trip Hour:',hour_count.idxmin())
print('Lowest Number of Trips:',hour_count.min())
plt.figure(figsize=(10,6))
plt.bar(hour_count.index,hour_count.values,color='yellow',width=0.7)
plt.xticks(range(24),rotation=45)
plt.xlabel('Pickup Hour')
plt.ylabel('Number of Trips')
plt.title('Trips by Hour')
plt.show()

In [ ]:
print('='*60)
print('Trips by Day')
print('='*60)
days=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_count=clean_df['day_of_week'].value_counts()
for day in days:
  print(f'{day} : {day_count[day]}')
print('\nHighest Trip Day:',day_count[days].idxmax())
print('Highest Number of Trips:',day_count[days].max())
print('Lowest Trip Day:',day_count[days].idxmin())
print('Lowest Number of Trips:',day_count[days].min())
plt.figure(figsize=(10,6))
plt.bar(days,[day_count[day] for day in days],color='yellow',width=0.7)
plt.xticks(rotation=45)
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.title('Trips by Day of Week')
plt.show()

In [ ]:
print('='*60)
print('Trips by Hour and Day')
print('='*60)
days=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
hour_day=clean_df.pivot_table(
    index='day_of_week',
    columns='pickup_hour',
    values='tpep_pickup_datetime',
    aggfunc='count',
    fill_value=0
)
hour_day=hour_day.reindex(days)
print('Trips by Hour and Day:')
print(hour_day)
plt.figure(figsize=(14,7))
plt.imshow(hour_day,aspect='auto')
plt.xticks(range(24),range(24))
plt.yticks(range(7),days)
plt.xlabel('Pickup Hour')
plt.ylabel('Day of Week')
plt.title('Trips by Hour and Day')
plt.colorbar(label='Number of Trips')
plt.show()

In [ ]:
print('='*60)
print('Average Duration by Hour')
print('='*60)
avg_duration_hour=clean_df.groupby('pickup_hour')['trip_duration_minutes'].mean()
print('Average Trip Duration by Hour:')
print(avg_duration_hour)
print('\nHour with Longest Average Duration:',avg_duration_hour.idxmax())
print('Longest Average Duration:',avg_duration_hour.max())
plt.figure(figsize=(10,6))
plt.bar(avg_duration_hour.index,avg_duration_hour.values,color='yellow',width=0.7)
plt.xticks(range(24),rotation=45)
plt.xlabel('Pickup Hour')
plt.ylabel('Average Trip Duration (Minutes)')
plt.title('Average Trip Duration by Hour')
plt.show()

In [ ]:
print('='*60)
print('Average  Distance by Hour')
print('='*60)
avg_distance_hour=clean_df.groupby('pickup_hour')['trip_distance'].mean()
print('Average Trip Distance by Hour:')
print(avg_distance_hour)
print('\nHour with Highest Average Distance:',avg_distance_hour.idxmax())
print('Highest Average Distance:',avg_distance_hour.max())
plt.figure(figsize=(10,6))
plt.bar(avg_distance_hour.index,avg_distance_hour.values,color='yellow',width=0.7)
plt.xticks(range(24),rotation=45)
plt.xlabel('Pickup Hour')
plt.ylabel('Average Trip Distance (Miles)')
plt.title('Average Trip Distance by Hour')
plt.show()

# **CONCLUSIONS OF TIME BASED ANALYSIS**

**Which hour has the highest number of trips?**

Highest Trip Hour: 17

**Which hour has the lowest number of trips?**

Lowest Trip Hour: 4

**Which day of the week has the highest number of trips?**

Highest Trip Day: Thursday

**Which day has the lowest number of trips?**

Lowest Trip Day: Monday

**How does weekday activity compare with weekend activity?**

Weekend Activity is more

**Which hours have the longest average trip duration?**

Longest Average Duration: 16.8389633989616

**Which hours have the highest average trip distance?**

Hour with Longest Average Duration: 16

**Does average speed change throughout the day?**

Yes


# **TRIP DISTANCE AND DURATION ANALYSIS**

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(clean_df['trip_distance'],clean_df['trip_duration_minutes'],alpha=0.3,color='yellow')
plt.xlabel('Trip Distance (Miles)')
plt.ylabel('Trip Duration (Minutes)')
plt.title('Trip Distance vs Trip Duration')
plt.show()


# **CONCLUSIONS OF TRIP DISTANCE AND DURATION ANALYSIS**
**What is the average trip distance?**

Average Trip Distance: 3.243206340880859

**What is the median trip distance?**

Median Trip Distance: 1.62

**What is the average trip duration?**

Average Trip Duration: 15.099007664312422

**What is the median trip duration?**

Median Trip Duration: 11.35

**Do longer trips generally take longer?**

Yes

**Are there trips with very high distance but unusually low duration?**

Yes

**Are there trips with very low distance but unusually high duration?**

Yes

 **What does this tell you about potential data anomalies?**

Extreme distance-duration values may indicate incorrect timestamps, unrealistic speeds, or other data anomalies. These records should be investigated before removal.


# **PASSENGER ANALYSIS**

In [ ]:
print('='*60)
print('Passenger Analysis')
print('='*60)
passenger_count=clean_df['passenger_count'].value_counts().sort_index()
print('Passenger Count Distribution:')
print(passenger_count)
print('\n1. Most Common Passenger Count:',clean_df['passenger_count'].mode()[0])
one_passenger=(clean_df['passenger_count']==1).sum()
total_trips=clean_df['passenger_count'].notna().sum()
print('2. Percentage of Trips with One Passenger:',(one_passenger/total_trips)*100)
plt.figure(figsize=(10,6))
plt.bar(passenger_count.index.astype(str),passenger_count.values,color='yellow',width=0.7)
plt.xlabel('Passenger Count')
plt.ylabel('Number of Trips')
plt.title('Passenger Count Distribution')
plt.show()

In [ ]:
print('='*60)
print('Trip Distance by Passenger Count')
print('='*60)
print(clean_df.groupby('passenger_count')['trip_distance'].mean())
plt.figure(figsize=(10,6))
clean_df.boxplot(column='trip_distance', by='passenger_count')
plt.xlabel('Passenger Count')
plt.ylabel('Trip Distance (Miles)')
plt.title('Trip Distance by Passenger Count')
plt.suptitle('')
plt.show()

In [ ]:
print('='*60)
print('Passenger Count: Weekday vs Weekend')
print('='*60)
passenger_day=clean_df.groupby(['is_weekend','passenger_count']).size().unstack(fill_value=0)
print(passenger_day)
passenger_day.index=['Weekday','Weekend']
passenger_day.plot(kind='bar',figsize=(10,6),color='yellow')
plt.xlabel('Day Type')
plt.ylabel('Number of Trips')
plt.title('Passenger Count: Weekday vs Weekend')
plt.xticks(rotation=0)
plt.show()

# **CONCLUSIONS OF PASSENGER COUNT**



**What is the most common passenger count?**

Most Common Passenger Count: 1.0

**What percentage of trips have one passenger?**

Percentage of Trips with One Passenger: 79.81191617794616

**How does trip distance vary with passenger count?**

Trip distance varies slightly with passenger count.

 **How does trip duration vary with passenger count?**

Trip duration also varies slightly with passenger count.

**Are passenger counts distributed differently during weekdays and weekends?**

Yes


# **PICKUP AND DROP LOCATION ANALYSIS**

In [ ]:
print('='*60)
print('TOP 10 PICKUP LOCATIONS')
print('='*60)
pickup_count=clean_df['PULocationID'].value_counts().head(10)
print('\n1. TOP 10 PICKUP LOCATIONS')
print(pickup_count)
plt.figure(figsize=(10,6))
plt.bar(pickup_count.index.astype(str),pickup_count.values,color='yellow')
plt.xlabel('Pickup Location ID')
plt.ylabel('Number of Trips')
plt.title('Top 10 Pickup Locations')
plt.show()

In [ ]:
print('='*60)
print('PICKUP LOACATIONS WITH HIGHEST NUMBER OF TRIPS')
print('='*60)
print('\n2. PICKUP LOCATIONS WITH HIGHEST NUMBER OF TRIPS')
print(pickup_count)

In [ ]:
print('='*60)
print('TOP 10 DROP LOCATIONS')
print('='*60)
dropoff_count=clean_df['DOLocationID'].value_counts().head(10)
print('\n3. TOP 10 DROP-OFF LOCATIONS')
print(dropoff_count)
plt.figure(figsize=(10,6))
plt.bar(dropoff_count.index.astype(str),dropoff_count.values,color='yellow')
plt.xlabel('Drop-off Location ID')
plt.ylabel('Number of Trips')
plt.title('Top 10 Drop-off Locations')
plt.show()

In [ ]:
print('='*60)
print('HIGHEST TRIP RECEIVING DROP-OFF LOCATIONS')
print('='*60)
print('\n4. DROP-OFF LOCATIONS WITH HIGHEST NUMBER OF TRIPS')
print(dropoff_count)

In [ ]:
print('='*60)
print(' MOST FREQUENTLY TRAVELLED ROUTES')
print('='*60)
route_count=( clean_df.groupby(['PULocationID','DOLocationID']) .size() .reset_index(name='Number of Trips') .sort_values('Number of Trips',ascending=False)    .head(10))
print('\n5. TOP 10 MOST FREQUENTLY TRAVELLED ROUTES')
print(route_count.to_string(index=False))

In [ ]:

print('='*60)
print('COMMON PICKUP AND DROP-OFF LOCATIONS')
print('='*60)
top_pickups=set(pickup_count.index)
top_dropoffs=set(dropoff_count.index)
common_locations=top_pickups.intersection(top_dropoffs)
print('\n6. COMMON PICKUP AND DROP-OFF LOCATIONS')
print(sorted(common_locations))
print('Number of common locations:',len(common_locations))

In [ ]:
print('='*60)
print('ROUTES WITH LONGEST AVERAGE TRIP DISTANCE')
print('='*60)
route_distance=(clean_df.groupby(['PULocationID','DOLocationID']).agg(Average_Trip_Distance=('trip_distance','mean'),Number_of_Trips=('trip_distance','size')) .sort_values('Average_Trip_Distance',ascending=False).head(10))
print('\n7. ROUTES WITH LONGEST AVERAGE TRIP DISTANCE')
print(route_distance)


In [ ]:
print('='*60)
print('ROUTES WITH LONGEST AVERAGE TRIP DURATION')
print('='*60)
route_duration=(clean_df.groupby(['PULocationID','DOLocationID']).agg(Average_Trip_Duration=('trip_duration_minutes','mean'), Number_of_Trips=('trip_duration_minutes','size')) .sort_values('Average_Trip_Duration',ascending=False).head(10))
print('\n8. ROUTES WITH LONGEST AVERAGE TRIP DURATION')
print(route_duration)


# **PICK UP AND DROP LOCATION ANALYSIS**

**PICK UP ANALYSIS**

**What are the top 10 pickup locations?**

237, 161, 132  ,  236  , 186  ,  230   , 162   , 142  ,  138   ,  163   

**Which pickup locations generate the highest number of trips?**

237, 161, 132

**DROP ANALYSIS**

**What are the top 10 drop-off locations?**

236, 237 ,161 ,230     ,142    ,239    ,170    ,141     ,162, 163

**Which drop-off locations receive the highest number of trips?**

236, 237, 161

**ROUTE ANALYSIS**



**Are the most common pickup locations also the most common drop-off locations?**

Some of the locations are common    142, 161, 162, 163, 230, 236, 237

**Which routes have the longest average trip distance?**

265 → 101,
61 → 47,
3 → 100,
72 → 140,
265 → 85,
247 → 265,
101 → 265,
265 → 75,
197 → 265,
54 → 265,

**Which routes have the longest average trip duration?**

76 → 132,
108 → 13,
125 → 208,
90 → 205,
255 → 193,
41 → 193,
149 → 65,
195 → 172,
249 → 196,
255 → 157

# **PAYMENT TYPE ANALYSIS**

In [ ]:
print('='*60)
print('PAYMENT TYPE COUNT PLOT ')
print('='*60)
plt.figure(figsize=(10,6))
plt.bar(payment_count.index.astype(str),payment_count.values,color='yellow',width=0.7)
plt.xlabel('Payment Type')
plt.ylabel('Number of Trips')
plt.title('Payment Type Count')
plt.show()

In [ ]:
print('='*60)
print('PAYMENT TYPE PERCENTAGE ')
print('='*60)
plt.figure(figsize=(10,6))
plt.bar(payment_percentage.index.astype(str),payment_percentage.values,color='yellow',width=0.7)
plt.xlabel('Payment Type')
plt.ylabel('Percentage of Trips')
plt.title('Payment Type Percentage')
plt.show()


In [ ]:
print('='*60)
print('PAYMENT TYPE VS TRIP DISTANCE ')
print('='*60)
plt.figure(figsize=(10,6))
plt.bar(distance_payment.index.astype(str),distance_payment.values,color='yellow',width=0.7)
plt.xlabel('Payment Type')
plt.ylabel('Average Trip Distance (Miles)')
plt.title('Payment Type vs Trip Distance')
plt.show()


In [ ]:
print('='*60)
print('PAYMENT TYPE VS TRIP DURATION ')
print('='*60)
plt.figure(figsize=(10,6))
plt.bar(duration_payment.index.astype(str),duration_payment.values,color='yellow',width=0.7)
plt.xlabel('Payment Type')
plt.ylabel('Average Trip Duration (Minutes)')
plt.title('Payment Type vs Trip Duration')
plt.show()

# **CONCLUSIONS OF PAYMENT TYPE ANALYSIS**

**What are the different payment types present?**

[1, 2, 3, 4]

**Which payment type occurs most frequently?**

1

**What percentage of trips belong to each payment type?**


1     ->			83.74,
2   	->		13.15,
3   ->		 	 0.64,
4  		-> 	 2.47,

**Does trip distance vary across payment types?**

Trip distance varies across payment types, with some payment types having higher average distances.

**Does trip duration vary across payment types?**

Trip duration also varies across payment types, with some payment types having longer average trip durations.

**Does passenger count vary across payment types?**

Passenger count varies slightly across payment types, although the differences are not very large.


# **VENDOR ANALYSIS**

In [ ]:
print('='*60)
print('TRIPS RECORDED BY EACH VENDOR')
print('='*60)
vendor_count=clean_df['VendorID'].value_counts().sort_index()
print('\n1. Trips by Vendor:')
print(vendor_count)


In [ ]:
print('='*60)
print('PERCENTAGE OF TRIPS BY EACH VENDOR')
print('='*60)
vendor_percentage=(vendor_count/len(clean_df)*100).round(2)
print('\n2. Percentage of Trips:')
print(vendor_percentage)

In [ ]:
print('='*60)
print('AVERGAE TRIP DISTANCE BY VENDOR')
print('='*60)
average_distance=clean_df.groupby('VendorID')['trip_distance'].mean()
print('\n3. Average Trip Distance:')
print(average_distance)


In [ ]:
print('='*60)
print('AVERAGE TRIP DURATION BY VENDOR')
print('='*60)
average_duration=clean_df.groupby('VendorID')['trip_duration_minutes'].mean()
print('\n4. Average Trip Duration:')
print(average_duration)

In [ ]:
print('='*60)
print('PASSENGER COUNT DISTRIBUTION')
print('='*60)
passenger_distribution=pd.crosstab(clean_df['VendorID'],clean_df['passenger_count'])
print('\n5. Passenger Count Distribution:')
print(passenger_distribution)

In [ ]:
print('='*60)
print('PAYMENT TYPE DISTRIBUTION')
print('='*60)
payment_distribution=pd.crosstab(clean_df['VendorID'],clean_df['payment_type'])
print('\n6. Payment Type Distribution:')
print(payment_distribution)


# **CONCLUSIONS OF VENDOR ANALYSIS**

**How many trips are recorded by each vendor?**


1    	->	 628844,
2    	-> 	2242429

**What percentage of trips does each vendor represent?**


1    	->	21.9,
2    	->	78.1

**Do vendors have different average trip distances?**

Yes

**Do vendors have different average trip durations?**

Yes

**Do vendors show different passenger-count distributions?**

Yes

**Are there differences in payment-type distribution between vendors?**

Yes

**If you identify differences between vendors, describe the observed difference.**

Vendors show differences in trip volume, average distance, duration, passenger count, and payment-type distribution. These differences indicate variation in trip patterns, but they do not determine which vendor performs better.


# **RATE CODE ANALYSIS**

In [ ]:
print('='*60)
print('MOST FREQUENTLY USED RATE CODES')
print('='*60)
rate_count=clean_df['RatecodeID'].value_counts().sort_index()
print('\n1. Rate Code Frequency:')
print(rate_count)

In [ ]:
print('='*60)
print('PERCENTAGE OF TRIPS BY RATE CODE')
print('='*60)
rate_percentage=(rate_count/len(clean_df)*100).round(2)
print('\n2. Percentage of Trips:')
print(rate_percentage)

In [ ]:
print('='*60)
print('AVERAGE TRIP DISTANCE BY RATE CODE')
print('='*60)
rate_distance=clean_df.groupby('RatecodeID')['trip_distance'].mean()
print('\n3. Average Trip Distance by Rate Code:')
print(rate_distance)

In [ ]:
print('='*60)
print('AVERAGE TRIP DURATION BY RATE CODE')
print('='*60)
rate_duration=clean_df.groupby('RatecodeID')['trip_duration_minutes'].mean()
print('\n4. Average Trip Duration by Rate Code:')
print(rate_duration)



In [ ]:
print('='*60)
print('IDENTIFY UNUSUALLY LONG TRIPS')
print('='*60)
long_trip_threshold=clean_df['trip_duration_minutes'].quantile(0.99)
long_trips=clean_df[clean_df['trip_duration_minutes']>long_trip_threshold]
long_trip_rates=long_trips['RatecodeID'].value_counts()
print('\n5. Rate Codes Associated with Unusually Long Trips:')
print(long_trip_rates)


# **CONCLUSIONS ON RATE CODE ANALYSIS**

**Which rate codes are most frequently used?**

Rate Code 1

**What percentage of trips belongs to each rate code?**


1.0 -> 94.40,
2.0  ->3.17,
3.0  ->  0.28,
4.0  ->  0.25,
5.0  ->  0.55,
6.0 ->   0.00,
99.0   ->1.36

**How does trip distance vary across rate codes?**

Trip distance varies across rate codes, with some codes having longer average distances.

**How does trip duration vary across rate codes?**

Trip duration also varies across rate codes, with some codes showing longer average durations.

**Are some rate codes associated with unusually long trips?**

Yes

**Are there rare rate codes that require further investigation?**

Yes


# **STORE AND FORWARD ANALYSIS**

In [ ]:
print('='*60)
print('STORE AND FORWARD COUNT')
print('='*60)
flag_count=clean_df['store_and_fwd_flag'].value_counts()
print('\nStore and Forward Count:')
print(flag_count)
plt.figure(figsize=(8,6))
plt.bar(flag_count.index.astype(str),flag_count.values,color='yellow', width=0.7)
plt.xlabel('Store and Forward Flag')
plt.ylabel('Number of Trips')
plt.title('Store and Forward Trip Distribution')
plt.show()

In [ ]:
print('='*60)
print('TRIP DISTANCE COMPARISON')
print('='*60)
distance_flag=clean_df.groupby('store_and_fwd_flag')['trip_distance'].mean()
print('\nAverage Trip Distance:')
print(distance_flag)
plt.figure(figsize=(8,6))
plt.bar(distance_flag.index.astype(str),distance_flag.values,color='yellow',width=0.7)
plt.xlabel('Store and Forward Flag')
plt.ylabel('Average Trip Distance (Miles)')
plt.title('Trip Distance by Store and Forward Flag')
plt.show()

In [ ]:
print('='*60)
print('TRIP DURATION COMPARISON')
print('='*60)
duration_flag=clean_df.groupby('store_and_fwd_flag')['trip_duration_minutes'].mean()
print('\nAverage Trip Duration:')
print(duration_flag)
plt.figure(figsize=(8,6))
plt.bar(duration_flag.index.astype(str),duration_flag.values,color='yellow',width=0.7)
plt.xlabel('Store and Forward Flag')
plt.ylabel('Average Trip Duration (Minutes)')
plt.title('Trip Duration by Store and Forward Flag')
plt.show()

# **CONCLUSIONS OF STORE AND FORWARD ANALYSIS**

**What percentage of trips are marked as store-and-forward?**

N  -->  2864442,
Y   -->    6831

**How does trip distance vary between the two groups?**

N has more Average trip distance compared to Y

**How does trip duration vary between the two groups?**


N has more Average trip duration compared to Y

**Are there meaningful differences in the characteristics of these trips?**

Yes, There are meaningful Differences


# **NUMPY STATISTICAL ANALYSIS**

In [ ]:
print('='*60)
print('NumPy Statistical Analysis')
print('='*60)
columns=['trip_distance','trip_duration_minutes','passenger_count', 'average_speed_mph']
for column in columns:
    data=clean_df[column].dropna().to_numpy()
    print('\n'+'='*60)
    print(column)
    print('='*60)
    print('Mean:',np.mean(data))
    print('Median:',np.median(data))
    print('Minimum:',np.min(data))
    print('Maximum:',np.max(data))
    print('Variance:',np.var(data))
    print('Standard Deviation:',np.std(data))
    print('\nPercentiles:')
    print('25th:',np.percentile(data,25))
    print('50th:',np.percentile(data,50))
    print('75th:',np.percentile(data,75))
    print('90th:',np.percentile(data,90))
    print('95th:',np.percentile(data,95))
    print('99th:',np.percentile(data,99))

# **OUTLIER ANALYSIS USING IQR**

In [ ]:
print('='*60)
print('Outlier Analysis Using IQR')
print('='*60)
columns=['trip_distance','trip_duration_minutes','average_speed_mph']
for column in columns:
    data=clean_df[column].dropna()
    Q1=np.percentile(data,25)
    Q3=np.percentile(data,75)
    IQR=Q3-Q1
    lower_bound=Q1-1.5*IQR
    upper_bound=Q3+1.5*IQR
    outliers=data[(data<lower_bound)|(data>upper_bound)]
    print('\n'+'='*60)
    print(column)
    print('='*60)
    print('Q1:',Q1)
    print('Q3:',Q3)
    print('IQR:',IQR)
    print('Lower Bound:',lower_bound)
    print('Upper Bound:',upper_bound)
    print('Number of Outliers:',len(outliers))
    print('Outlier Percentage:',(len(outliers)/len(data))*100)

# **CORRELATION ANALYSIS**

In [ ]:
print('='*60)
print('Phase M - Correlation Analysis')
print('='*60)
columns=['passenger_count','trip_distance','trip_duration_minutes','average_speed_mph']
correlation_matrix=clean_df[columns].corr()
print('\nCorrelation Matrix:')
print(correlation_matrix)
plt.figure(figsize=(10,7))
sns.heatmap(correlation_matrix,annot=True,cmap='YlGn',fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

# **CONCLUSIONS OF CORRELATION ANALYSIS**

**Which variables have a strong relationship?**

Trip distance and trip duration have the strongest positive relationship among the selected variables.

**Which variables have a weak relationship?**

Passenger count has a weak relationship with trip distance, trip duration, and average speed.

** Is trip distance positively related to trip duration?**

Yes, trip distance is positively related to trip duration

**Is passenger count strongly related to trip distance?**

 No, passenger count is not strongly related to trip distance.


**Does correlation imply causation in this dataset?**


No. Correlation does not imply causation.

# **ADVANCED ANALYSIS**

In [ ]:
print('='*60)
print('DEMAND HEATMAP')
print('='*60)
day_order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
demand_heatmap=clean_df.pivot_table(index='day_of_week',columns='pickup_hour',values='tpep_pickup_datetime',aggfunc='count').reindex(day_order)
print(demand_heatmap)
plt.figure(figsize=(14,6))
sns.heatmap(demand_heatmap,annot=True,fmt='.0f',cmap='YlOrBr')
plt.xlabel('Pickup Hour')
plt.ylabel('Day of Week')
plt.title('Taxi Demand by Day and Hour')
plt.show()

In [ ]:
print('='*60)
print('TOP 20 MOST FREQUENT TRAVELLED ROUTES')
print('='*60)
top_20_routes=clean_df['route'].value_counts().head(20)
print(top_20_routes)
plt.figure(figsize=(12,7))
plt.barh(top_20_routes.index[::-1],top_20_routes.values[::-1],color='yellow')
plt.xlabel('Number of Trips')
plt.ylabel('Route')
plt.title('Top 20 Most Frequently Travelled Routes')
plt.show()

In [ ]:
print('='*60)
print(' ROUTE DURATION ANALYSIS')
print('='*60)
route_duration=clean_df.groupby('route').agg(trip_count=('route','size'),average_duration=('trip_duration_minutes','mean'))
route_duration=route_duration[route_duration['trip_count']>=100]
top_duration_routes=route_duration.sort_values('average_duration',ascending=False).head(20)
print(top_duration_routes)

In [ ]:
print('='*60)
print('SPEED ANALYSIS')
print('='*60)
speed=clean_df['average_speed_mph'].dropna()
print('Average Speed:',speed.mean())
print('Median Speed:',speed.median())
print('Minimum Speed:',speed.min())
print('Maximum Speed:',speed.max())
print('\nSpeed Percentiles:')
print(speed.quantile([0.01,0.25,0.50,0.75,0.95,0.99]))
plt.figure(figsize=(10,6))
plt.hist(speed,bins=50,color='yellow')
plt.xlabel('Average Speed (mph)')
plt.ylabel('Number of Trips')
plt.title('Average Speed Distribution')
plt.show()

In [ ]:
print('='*60)
print('DISTANCE BAND ANALYSIS')
print('='*60)
clean_df['distance_band']=pd.cut(clean_df['trip_distance'],bins=[0,5,10,float('inf')],labels=['Short','Medium','Long'],include_lowest=True)
distance_analysis=clean_df.groupby('distance_band',observed=True).agg(trip_count=('trip_distance','size'),average_duration=('trip_duration_minutes','mean'), average_speed=('average_speed_mph','mean'),average_passenger_count=('passenger_count','mean'))
print(distance_analysis)

In [ ]:
print('='*60)
print('PEAK VS OFF-PEAK ANALYSIS')
print('='*60)
peak_hours=[7,8,9,10,16,17,18,19,20]
clean_df['period']=np.where(clean_df['pickup_hour'].isin(peak_hours),'Peak','Off-Peak')
peak_analysis=clean_df.groupby('period').agg(trip_count=('period','size'),average_distance=('trip_distance','mean'),average_duration=('trip_duration_minutes','mean'),average_speed=('average_speed_mph','mean'))
print(peak_analysis)

In [ ]:
print('='*60)
print('ROUTE CONCENTRATION')
print('='*60)
route_counts=clean_df['route'].value_counts()
total_trips=len(clean_df)
for n in [10,20,50]:
    top_trips=route_counts.head(n).sum()
    percentage=(top_trips/total_trips)*100
    print(f'Top {n} Routes:',round(percentage,2),'% of total trips')

# **BUSINESS QUESTIONS**

**When is taxi demand highest?**

Taxi demand is highest during the busiest pickup hours, particularly during peak commuting periods.

**When is taxi demand lowest?**

Demand is lowest during the hours with the fewest recorded trips, generally during late-night or early-morning periods.

**How does weekday demand compare with weekend demand?**

Weekdays generally show higher and more consistent taxi demand, while weekends have different demand patterns across the day.

**What does a typical taxi trip look like in terms of distance and duration?**

A typical trip is relatively short in both distance and duration, while a smaller number of longer trips increase the overall averages.

**What are the most common pickup locations?**

The most common pickup locations are the locations with the highest number of recorded trips, indicating areas with strong passenger demand.

**What are the most common drop-off locations?**

The most common drop-off locations are the locations receiving the highest number of taxi trips.

**What are the most frequently travelled routes?**

The most frequently travelled routes include 237 → 236, 236 → 237, 236 → 236, and 237 → 237.

**Are longer trips generally associated with longer durations?**

Yes. Longer trip distances generally tend to require longer trip durations.

**Which hours show unusually long or short trip durations?**

Average trip duration varies by hour, with some hours showing unusually long or short trips that may reflect traffic conditions, demand patterns, or unusual records.

**What passenger-count patterns can be observed?**

Most trips involve a small number of passengers, with one passenger being the most common passenger count.

**Which payment types are most common?**

Payment Type 1 is the most frequently used payment type and
represents the majority of trips.

**How do trip characteristics vary across vendors?**

Vendors differ in trip volume, average distance, average duration, passenger-count distribution, and payment-type distribution. Vendor 2 records the highest trip volume.

**Which rate codes are most frequently used?**

Rate Code 1 is the most frequently used rate code, followed by the other less frequently used codes.

**What unusual or potentially problematic records exist?**

Potential issues include extremely large trip distances, unusually long durations, unrealistic speeds, invalid passenger counts, drop-offs occurring before pickups, and zero-duration trips.

**What are the most important data-quality limitations?**

The main limitations include extreme outliers, unusual timestamps, missing or invalid values, and potentially unrealistic distance or speed records. These records can affect averages and should be considered when interpreting the results.

**What operational questions could a taxi company or transportation analyst investigate next?**

A taxi company could investigate where and when demand is highest, how to improve vehicle allocation, which routes generate the most trips, how peak-hour traffic affects duration, and how demand patterns could support better fleet planning.

# **FINAL KPI SUMMARY**



# **KEY FINDINGS**

1. Most taxi trips are short,with the average trip covering about 3.24 miles.

2. A typical taxi trip lasts around 11 minutes,while the overall average is about 15 minutes.

3. Taxi demand is highest around 5 PM,which suggests strong demand during the evening travel period.

4. Thursday recorded the highest number of trips among the days of the week.
5. Most trips are taken by one passenger,making single-passenger rides the most common.
6. Payment Type 1 is clearly the most commonly used payment method.
7. Location 237 is the most common pickup location,while Location 236 is the most common drop-off location.
8. The most frequently travelled route is 237 → 236,with 23,697 trips.
9. Longer trips generally take longer to complete,showing a clear positive relationship between trip distance and trip duration.
10. Different vendors show different travel patterns,including differences in trip volume,distance,duration,and payment methods. Vendor 2 recorded the highest number of trips.
11. Rate Code 1 is used most frequently,while some rate codes are relatively rare and may need further investigation.
12. The data contains some unusual trips,such as extremely long distances,very long durations,and unusually high speeds.
13. Some unusual speed values may be caused by incorrect timestamps,very short trip durations,unusual distances,or other data-quality issues.
14. The analysis found several data-quality problems,including invalid distances,invalid passenger counts,zero-duration trips,and drop-off times occurring before pickup times.
15. Overall,the data shows that taxi demand is concentrated around specific times,locations,and routes,which could help taxi companies improve vehicle allocation and service planning.

# **BUSINESS INSIGHTS**

1. Taxi demand is highest around 5 PM,so more taxis should be available during the evening peak period.

2. Thursday has the highest trip demand,which can help in planning driver availability for busy days.
3. Since most trips are short-distance trips,taxis can focus on serving high-demand areas rather than only long-distance journeys.
4. Locations 237 and 236 generate and receive a large number of trips,so these areas may require more taxi availability.
5. The 237 → 236 route has the highest number of trips,indicating a strong and consistent travel pattern between these locations.
6. Trip demand changes throughout the day,so driver and vehicle allocation should be adjusted based on demand by hour.
7. Since most trips have one passenger,the majority of trips are suitable for standard taxi services rather than larger vehicles.
8. The dominance of Payment Type 1 suggests that this payment method should be well supported and reliable.
9. Differences between vendors in trip volume,distance,and duration can help companies compare operating patterns and identify areas for improvement.
10. Routes with unusually long trip durations may indicate traffic congestion,delays,or unusual trips,so these routes should be investigated.
11. Unusually high speeds and extreme trip distances should be reviewed because they may affect operational reports,performance analysis,and business decisions.
12. The analysis can help taxi operators improve fleet allocation,driver scheduling,peak-hour availability,and service coverage.
13. High-demand pickup and drop-off locations can be used to position taxis closer to customers and potentially reduce waiting time.
14. Historical trip patterns can help businesses forecast demand and plan resources more effectively.
15. Overall,the analysis provides useful information for improving taxi availability,route planning,driver allocation,customer service,and operational efficiency.

# **LIMITATIONS**

1. The analysis is based on one month of taxi trip data,so it may not capture seasonal or long-term trends.

2. Extreme values in distance,duration,and speed can influence averages and other statistical results.
3. Some records contain missing or unusual values,which may reduce the accuracy of certain analyses.
4. The dataset does not include detailed information about traffic congestion,so the exact reasons for longer trip durations cannot be confirmed.
5. Weather conditions are not included,so their impact on taxi demand cannot be measured.
6. The data does not provide detailed information about driver availability or working hours,limiting workforce analysis.
7. Passenger demand may vary because of events,holidays,or special occasions,which are not considered in the analysis.
8. Location IDs are used instead of detailed location names,which makes it harder to explain the exact areas represented by each location.
9. Some rare values may represent valid trips,but they require additional business or domain information to interpret correctly.
10. The findings describe patterns in the available data,but they should not be treated as definitive predictions of future taxi demand.

# **FINAL CONCLUSION**

The Yellow Taxi dataset analysis provided a clear understanding of taxi demand,trip patterns,passenger behavior,payment methods,and route activity.

Most trips were short and involved a single passenger,with an average trip distance of 3.24 miles and an average duration of 15.10 minutes. Demand was highest around 5 PM,while Thursday was the busiest day.

The analysis also showed that certain locations and routes receive significantly more activity,with Location 237 being the most common pickup location,Location 236 being the most common drop-off location,and 237 → 236 being the most frequently travelled route.

The data also revealed some unusual distances,durations,and speeds,highlighting the importance of data cleaning and outlier investigation before making business decisions.

Overall,the analysis demonstrates how taxi trip data can be used to understand customer demand,identify high-traffic locations,plan vehicle availability,optimize routes,and improve operational efficiency. Future analysis using data from multiple months,weather,traffic,and other external factors could provide even more accurate business insights.